In [5]:
# =========================================================
# INDOFLOODS - INFRASTRUCTURE & POPULATION DATA SCORE
# Python 3.7 compatible
# =========================================================

# Install openpyxl if needed
import sys
import subprocess

try:
    import openpyxl
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "openpyxl"
    ])

import pandas as pd
import numpy as np


# =========================================================
# 1. LOAD DATA
# =========================================================

FILE = "Final_INDOFLOODS_dataset.xlsx"

df = pd.read_excel(
    FILE,
    engine="openpyxl"
)

print("Dataset loaded successfully")
print("Rows    :", len(df))
print("Columns :", len(df.columns))


# =========================================================
# 2. INFRASTRUCTURE COLUMNS
# =========================================================

infra_cols = [
    "hospital_count_5km",
    "school_count_5km",
    "relief_shelter_count_5km"
]

# Use only columns that exist in the dataset
infra_cols = [
    col for col in infra_cols
    if col in df.columns
]

print("\nInfrastructure columns found:")
print(infra_cols)


# =========================================================
# 3. INFRASTRUCTURE SCORE
# =========================================================

infra_scores = []

for col in infra_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # -----------------------------------------------------
    # A. COMPLETENESS - 30%
    # -----------------------------------------------------

    completeness = values.notna().mean()

    # -----------------------------------------------------
    # B. VALIDITY - 25%
    # Infrastructure count cannot be negative
    # -----------------------------------------------------

    valid = (
        values.notna()
        &
        (values >= 0)
    )

    if values.notna().sum() > 0:
        validity = (
            valid.sum()
            /
            values.notna().sum()
        )
    else:
        validity = 0

    # -----------------------------------------------------
    # C. CONSISTENCY - 20%
    # Compare count with corresponding list column
    # -----------------------------------------------------

    name_col = col.replace(
        "_count_5km",
        "s_within_5km"
    )

    if name_col in df.columns:

        names = df[name_col]

        checked = 0
        consistent = 0

        for count, name in zip(values, names):

            if pd.isna(count):
                continue

            checked += 1

            text = str(name).strip().lower()

            # Count = 0 should normally mean none found
            if count == 0:

                if text in [
                    "none found",
                    "0",
                    "",
                    "nan"
                ]:
                    consistent += 1

            # Count > 0 should have actual names
            elif count > 0:

                if text not in [
                    "none found",
                    "data unavailable",
                    "",
                    "nan"
                ]:
                    consistent += 1

        if checked > 0:
            consistency = consistent / checked
        else:
            consistency = 0

    else:

        # If corresponding name column doesn't exist
        consistency = 0.80


    # -----------------------------------------------------
    # D. DUPLICATE HANDLING - 10%
    # -----------------------------------------------------

    if len(df) > 0:

        duplicate_score = (
            len(df.drop_duplicates())
            /
            len(df)
        )

    else:
        duplicate_score = 0


    # -----------------------------------------------------
    # E. SOURCE RELIABILITY - 15%
    #
    # Conservative score; not claiming 100%
    # -----------------------------------------------------

    source_reliability = 0.85


    # -----------------------------------------------------
    # FINAL INFRASTRUCTURE SCORE
    # -----------------------------------------------------

    score = (
        completeness * 0.30
        +
        validity * 0.25
        +
        consistency * 0.20
        +
        duplicate_score * 0.10
        +
        source_reliability * 0.15
    )

    score_percentage = score * 100
    score_out_of_10 = score * 10

    infra_scores.append(score_percentage)

    print("\n{}".format(col))
    print("  Completeness : {:.2f}%".format(completeness * 100))
    print("  Validity     : {:.2f}%".format(validity * 100))
    print("  Consistency  : {:.2f}%".format(consistency * 100))
    print("  Score        : {:.2f}/10".format(score_out_of_10))
    print("  Percentage   : {:.2f}%".format(score_percentage))


# Overall infrastructure score
if len(infra_scores) > 0:
    infrastructure_score = np.mean(infra_scores)
else:
    infrastructure_score = 0


# =========================================================
# 4. POPULATION SCORE
# =========================================================

pop_col = "population_within_5km"

if pop_col in df.columns:

    population = pd.to_numeric(
        df[pop_col],
        errors="coerce"
    )

    # -----------------------------------------------------
    # A. COMPLETENESS

Dataset loaded successfully
Rows    : 4548
Columns : 30

Infrastructure columns found:
['hospital_count_5km', 'school_count_5km', 'relief_shelter_count_5km']

hospital_count_5km
  Completeness : 100.00%
  Validity     : 100.00%
  Consistency  : 100.00%
  Score        : 9.78/10
  Percentage   : 97.75%

school_count_5km
  Completeness : 100.00%
  Validity     : 100.00%
  Consistency  : 36.04%
  Score        : 8.50/10
  Percentage   : 84.96%

relief_shelter_count_5km
  Completeness : 100.00%
  Validity     : 100.00%
  Consistency  : 6.53%
  Score        : 7.91/10
  Percentage   : 79.06%
